# Ejercicio 7: Predicción de Lluvia con XGBoost

Este notebook implementa un modelo de clasificación para predecir si lloverá mañana basándose en datos meteorológicos de Australia. Compararemos el rendimiento de un árbol de decisión tradicional con XGBoost, un algoritmo de gradient boosting más avanzado.

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier

import sklearn.metrics as metrics
from sklearn.model_selection import learning_curve



## 1. Importación de Librerías

Importamos todas las librerías necesarias para el análisis:
- **NumPy y Pandas**: Para manipulación de datos
- **Matplotlib y Seaborn**: Para visualizaciones
- **Scikit-learn**: Para preprocesamiento, división de datos, modelos y métricas de evaluación
- **DecisionTreeClassifier**: Modelo de árbol de decisión para comparación

In [23]:
data = pd.read_csv('weatherAUS.csv.zip')

## 2. Carga de Datos

Cargamos el dataset `weatherAUS.csv.zip` que contiene información meteorológica histórica de Australia. Este dataset incluye variables como temperatura, humedad, presión, dirección del viento y si llovió o no.

In [24]:
data.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


### 2.1 Exploración Inicial

Visualizamos las primeras filas del dataset para entender su estructura y las variables disponibles.

In [25]:
columnas_descartables = ['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am', 'Location', 'Date']
data = data.drop(columns=columnas_descartables)
data = data.dropna()    

## 3. Preprocesamiento de Datos

### 3.1 Eliminación de Columnas con Muchos Valores Faltantes

Eliminamos columnas que tienen demasiados valores nulos o que no son relevantes para nuestro análisis:
- **Sunshine** y **Evaporation**: Muchos valores faltantes
- **Cloud3pm** y **Cloud9am**: Información incompleta
- **Location** y **Date**: No las usaremos como features

Después eliminamos todas las filas con valores nulos restantes para tener un dataset limpio.

In [26]:
columnas_descartables = ['WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']
data = data.drop(columns=columnas_descartables)

### 3.2 Eliminación de Variables Categóricas

Eliminamos las variables de dirección del viento y la variable RainToday, ya que trabajaremos solo con variables numéricas para este ejercicio:
- **WindGustDir**, **WindDir9am**, **WindDir3pm**: Variables categóricas de dirección
- **RainToday**: Variable categórica que podría causar data leakage

Las variables a borrar son las co-relacionadas

In [27]:
data = data.drop(columns=['Temp3pm', 'Pressure9am'])


### 3.3 Eliminación de Variables Correlacionadas

Para evitar multicolinealidad en nuestro modelo, eliminamos variables que están altamente correlacionadas con otras:
- **Temp3pm**: Correlacionada con MaxTemp
- **Pressure9am**: Correlacionada con otras variables de presión

In [28]:
data['RainTomorrow']=data['RainTomorrow'].map({'No':0,'Yes':1})

### 3.4 Codificación de la Variable Objetivo

Convertimos la variable objetivo **RainTomorrow** de categórica (Yes/No) a numérica (1/0):
- **No** → 0 (no lloverá mañana)
- **Yes** → 1 (lloverá mañana)

In [29]:
columnas_entrenamiento = ['MaxTemp', 'Humidity3pm']
X=data[columnas_entrenamiento]
y=data['RainTomorrow']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


## 4. Preparación de Datos para Entrenamiento

### 4.1 Selección de Features y División de Datos

Seleccionamos solo dos variables para el entrenamiento:
- **MaxTemp**: Temperatura máxima del día
- **Humidity3pm**: Humedad a las 3 PM

Dividimos los datos en conjuntos de entrenamiento (70%) y prueba (30%), usando `stratify=y` para mantener la proporción de clases en ambos conjuntos.

In [30]:
from sklearn.tree import DecisionTreeClassifier


## 5. Modelo Baseline: Árbol de Decisión

### 5.1 Importación del Modelo

In [31]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

### 5.2 Entrenamiento del Árbol de Decisión

Creamos y entrenamos un modelo de árbol de decisión básico con configuración por defecto. Este servirá como modelo baseline para comparar con XGBoost.

In [32]:
y_pred_dt = dt_model.predict(X_test)

### 5.3 Predicciones del Árbol de Decisión

Generamos predicciones sobre el conjunto de prueba usando el modelo entrenado.

In [33]:
accuracy_score(y_test, y_pred_dt)

0.8045929511777554

### 5.4 Evaluación del Árbol de Decisión

Evaluamos el rendimiento del modelo usando tres métricas clave:

#### 5.4.1 Accuracy (Exactitud)
Proporción de predicciones correctas sobre el total de predicciones.

In [34]:
classification_report(y_test, y_pred_dt)

'              precision    recall  f1-score   support\n\n           0       0.83      0.94      0.88     26372\n           1       0.60      0.34      0.44      7506\n\n    accuracy                           0.80     33878\n   macro avg       0.72      0.64      0.66     33878\nweighted avg       0.78      0.80      0.78     33878\n'

#### 5.4.2 Classification Report
Reporte detallado que incluye:
- **Precision**: De las predicciones positivas, cuántas fueron correctas
- **Recall**: De los casos positivos reales, cuántos fueron detectados
- **F1-Score**: Media armónica entre precision y recall
- **Support**: Número de observaciones de cada clase

In [35]:
confusion_matrix(y_test, y_pred_dt)

array([[24671,  1701],
       [ 4919,  2587]])

#### 5.4.3 Matriz de Confusión
Muestra la distribución de predicciones:
- **Verdaderos Negativos (TN)**: No llueve y predijo que no llueve
- **Falsos Positivos (FP)**: No llueve pero predijo que llueve
- **Falsos Negativos (FN)**: Llueve pero predijo que no llueve
- **Verdaderos Positivos (TP)**: Llueve y predijo que llueve

In [36]:
from xgboost import XGBClassifier

## 6. Modelo XGBoost

### 6.1 Importación de XGBoost

XGBoost (Extreme Gradient Boosting) es un algoritmo de ensemble que combina múltiples árboles de decisión débiles para crear un modelo más robusto. Utiliza gradient boosting, donde cada árbol nuevo intenta corregir los errores de los árboles anteriores.

In [38]:
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)


### 6.2 Entrenamiento y Predicción con XGBoost

Creamos un modelo XGBoost con:
- `eval_metric='logloss'`: Métrica de evaluación logarítmica para clasificación
- `random_state=42`: Para reproducibilidad

Entrenamos el modelo con los mismos datos de entrenamiento y generamos predicciones.

In [39]:
accuracy_score(y_test, y_pred_xgb)

0.8300076745970837

### 6.3 Evaluación de XGBoost

Evaluamos el modelo XGBoost usando las mismas métricas que el árbol de decisión para poder comparar su rendimiento.

#### 6.3.1 Accuracy de XGBoost

In [40]:
classification_report(y_test, y_pred_xgb)

'              precision    recall  f1-score   support\n\n           0       0.84      0.96      0.90     26372\n           1       0.74      0.36      0.48      7506\n\n    accuracy                           0.83     33878\n   macro avg       0.79      0.66      0.69     33878\nweighted avg       0.82      0.83      0.81     33878\n'

#### 6.3.2 Classification Report de XGBoost

In [41]:
confusion_matrix(y_test, y_pred_xgb)

array([[25411,   961],
       [ 4798,  2708]])

## 7. Conclusiones

### Comparación de Modelos

Al comparar el Árbol de Decisión con XGBoost, podemos observar:

**Ventajas de XGBoost:**
- Mayor capacidad de generalización
- Mejor manejo de overfitting a través de regularización
- Implementa técnicas de ensemble que mejoran la robustez del modelo
- Generalmente produce mejores métricas de clasificación

**Consideraciones:**
- Ambos modelos utilizan las mismas features: MaxTemp y Humidity3pm
- La comparación permite evaluar la mejora que proporciona el gradient boosting sobre un árbol simple
- XGBoost suele ser más robusto ante outliers y datos ruidosos

### Próximos Pasos Potenciales:
1. Ajustar hiperparámetros de XGBoost (learning_rate, max_depth, n_estimators)
2. Incorporar más features para mejorar las predicciones
3. Realizar validación cruzada para evaluar la estabilidad del modelo
4. Analizar la importancia de las features
5. Probar con técnicas de balanceo de clases si existe desbalance

#### 6.3.3 Matriz de Confusión de XGBoost